In [13]:
import numpy as np
import polars as pl
from rapidfuzz import fuzz as rapidfuzz_fuzz, process

In [2]:
textos = pl.read_parquet('../database/textos/*.parquet').select('id', 'texto')

In [3]:
VALOR_PATTERN = r'R\$\s?(?:\d{1,3}(?:\.\d{3})*|\d+)(?:,\d{2})?'
ENT_PATTERN = r'[A-ZÁÂÃÉÊÍÓÔÕÚÇ]+\s[A-ZÁÂÃÉÊÍÓÔÕÚÇ\s]+'
PROCESSO_PATTERN = r'\b\d{5}\.?\d{6}\/?\d{4}\-?\d{2}\b'
CNPJ_PATTERN = r'\b\d{2}\.?\d{3}\.?\d{3}\/?\d{4}\-?\d{2}\b'
DATA_PATTERN = r'\b\d{1,2}/\d{1,2}/\d{2,4}\b|\b\d{1,2}\s+de\s+[a-zçãõáéíóúâêô]+\s+de\s+\d{4}\b'

In [4]:
def filter_nested_entities(items):
    if items is None:
        return []

    normalized_items = [item.strip() for item in items if item and item.strip()]

    return [
        item
        for item in normalized_items
        if not any(item != other and item in other for other in normalized_items)
    ]

textos = textos.with_columns(
    valores=pl.col('texto').str.extract_all(VALOR_PATTERN),
    entidades=pl.col('texto').str.extract_all(ENT_PATTERN).map_elements(
        filter_nested_entities,
        return_dtype=pl.List(pl.String)
    ),
    nups=pl.col('texto').str.extract_all(PROCESSO_PATTERN),
    cnpjs=pl.col('texto').str.extract_all(CNPJ_PATTERN),
    datas=pl.col('texto').str.to_lowercase().str.extract_all(DATA_PATTERN)
)

In [5]:
textos

id,texto,valores,entidades,nups,cnpjs,datas
str,str,list[str],list[str],list[str],list[str],list[str]
"""12002010442""","""PORTARIA DAC Nº 1.747/SIE, DE …",[],"[""PORTARIA DAC N"", ""DE DEZEMBRO DE"", … ""ALLEMANDER JESUS PEREIRA FILHO""]",[],[],"[""26 de dezembro de 2001"", ""21 de dezembro de 2001"", … ""20 de agosto de 1999""]"
"""12002010443""","""PORTARIA DAC Nº 1.748/SIE, DE …",[],"[""PORTARIA DAC N"", ""DE DEZEMBRO DE"", … ""ALLEMANDER JESUS PEREIRA FILHO""]",[],[],"[""26 de dezembro de 2001"", ""21 de dezembro de 2001"", … ""20 de agosto de 1999""]"
"""12002010444""","""PORTARIA DAC Nº 1.749/SIE, DE …",[],"[""PORTARIA DAC N"", ""DE DEZEMBRO DE"", … ""ALLEMANDER JESUS PEREIRA FILHO""]",[],[],"[""26 de dezembro de 2001"", ""21 de dezembro de 2001"", … ""20 de agosto de 1999""]"
"""12002010445""","""PORTARIA Nº 45/DAdM, DE 27 DE …",[],"[""PORTARIA N"", ""DE DEZEMBRO DE"", … ""ADOLF MAGNUS MONIZ OSTWALD""]",[],[],"[""27 de dezembro de 2001"", ""25 de julho de 2000"", … ""5 de janeiro de 2001""]"
"""12002010746""","""PORTARIA INTERMINISTERIAL Nº 2…",[],"[""PORTARIA INTERMINISTERIAL N"", ""DE JANEIRO DE"", … ""RONEY JOSE FERREIRA""]",[],[],"[""3 de janeiro de 2002"", ""11 de fevereiro de 2000"", … ""9 de fevereiro de 2001""]"
…,…,…,…,…,…,…
"""530_20251231_23469728""","""EXTRATO DE CREDENCIAMENTO Nº 3…","[""R$ 151.062,00""]","[""EXTRATO DE CREDENCIAMENTO N"", ""HOSPITAL NAVAL DE NATAL"", … ""NÃO SE APLICA""]","[""63064.010971/2023-38""]","[""08.242.471/0001-18""]","[""15/12/2025"", ""14/12/2026"", … ""30/12/2025""]"
"""530_20251231_23470217""","""EXTRATO DE CREDENCIAMENTO Nº 3…","[""R$ 20.000,00""]","[""EXTRATO DE CREDENCIAMENTO N"", ""HOSPITAL NAVAL DE NATAL"", … ""NÃO SE APLICA""]","[""63064.010971/2023-38""]","[""60.143.972/0001-67""]","[""15/12/2025"", ""14/12/2026"", … ""30/12/2025""]"
"""530_20251231_23470438""","""EXTRATO DE CREDENCIAMENTO Nº 2…","[""R$ 20.000,00""]","[""EXTRATO DE CREDENCIAMENTO N"", ""HOSPITAL NAVAL DE NATAL"", … ""NÃO SE APLICA""]","[""63064.010971/2023-38""]","[""70.030.606/0001-55""]","[""09/12/2025"", ""08/12/2026"", … ""30/12/2025""]"


In [6]:
print(textos.filter(pl.col('valores').list.len() > 0).item(6, 'texto'))

PORTARIA Nº 739/MD, DE 26 DE NOVEMBRO DE 2001
O MINISTRO DE ESTADO DA DEFESA, no uso da atribuição que lhe é conferida pelo art. 87, inciso IV, da Constituição Federal e da delegação de competência de trata o art. 1º do Decreto nº 3.439, de 25 de abril de 2000, tendo em vista o que consta dos processos nº 07-01/95218/00 - DAC e 60041.000029/01-19 - MD, resolve:
Art. 1º Fica concedida autorização à TRANSPORTES AÉREOS MERCANTILES PANAMERICANOS S.A., que poderá usar em sua razão social TAMPA S.A., com sede na cidade de Medellín, Colômbia, para funcionar no Brasil como empresa de transporte aéreo internacional regular de carga.
Art. 2º Ficam ainda estabelecidas as seguintes obrigações:
I - a TAMPA S.A. é obrigada a ter, permanentemente, um representante legal no Brasil, com plenos e ilimitados poderes para tratar e, definitivamente, resolver as questões que venham a surgir, quer com o Governo, quer com particulares, podendo ser demandado e receber citação inicial pela sociedade;
II - todos

In [7]:
print(textos.item(1, 'texto'))

PORTARIA DAC Nº 1.748/SIE, DE 26 DE DEZEMBRO DE 2001
Autoriza a empresa FREE HANDLING SERVIÇOS AUXILIARES DE TRANS-
PORTES AÉREOS LTDA a executar serviços auxiliares de transporte aéreo nos aeroportos brasileiros.
O CHEFE DO SUBDEPARTAMENTO DE INFRA-ESTRUTURA DO DEPARTAMENTO DE AVIAÇÃO CIVIL, no uso da delegação de competência outorgada pela Portaria DAC Nº 1727/DGAC, de 21 de dezembro de 2001, e de acordo com o art. 1º e 7º da Portaria nº 467/GM-5, de 3 de junho de 1993, e com fundamento no art. 102 da Lei nº 7565, de 19 de dezembro de 1986, que dispõe sobre o Código Brasileiro de Aeronáutica, resolve:
Art. 1º Autorizar a empresa FREE HANDLING SERVIÇOS AUXILIARES DE TRANSPORTES AÉREOS LTDA, sediada na cidade de São Paulo, para executar serviços auxiliares de transporte aéreo, nos aeroportos brasileiros, denominados Serviços Operacionais, discriminados no subitem 2.1 da Instrução de Aviação Civil IAC 2301-0899, aprovada pela Portaria nº 533/DGAC, de 12 de agosto de 1999, publicada no D

In [8]:
textos.item(1,  'entidades')

""
str
"""PORTARIA DAC N"""
"""DE DEZEMBRO DE"""
"""O CHEFE DO SUBDEPARTAMENTO DE …"
"""ESTRUTURA DO DEPARTAMENTO DE A…"
"""FREE HANDLING SERVIÇOS AUXILIA…"
"""ALLEMANDER JESUS PEREIRA FILHO"""


In [9]:
contagem_entidades = {}

for i in range(textos.height):
    for entidade in textos.item(i, 'entidades'):
        if entidade not in contagem_entidades:
            contagem_entidades[entidade] = 1
        else:
            contagem_entidades[entidade] += 1

In [10]:
contagem_entidades

{'PORTARIA DAC N': 1907,
 'DE DEZEMBRO DE': 20205,
 'JET SERVICE COMERCIAL LTDA': 2,
 'O CHEFE DO SUBDEPARTAMENTO DE INFRA': 1599,
 'ESTRUTURA DO DEPARTAMENTO DE AVIAÇÃO CIVIL': 1595,
 'ALLEMANDER JESUS PEREIRA FILHO': 1121,
 'FREE HANDLING SERVIÇOS AUXILIARES DE TRANSPORTES AÉREOS LTDA': 2,
 'SANPRESS SERVIÇOS AUXILIARES DO TRANSPORTE\nAÉREO LTDA': 1,
 'SANPRESS SERVIÇOS AUXILIARES DO TRANSPORTE AÉREO LTDA': 2,
 'PORTARIA N': 153949,
 'O DIRETOR DE ADMINISTRAÇÃO DA MARINHA': 537,
 'V A': 10346,
 'ADOLF MAGNUS MONIZ OSTWALD': 8,
 'PORTARIA INTERMINISTERIAL N': 180,
 'DE JANEIRO DE': 20544,
 'OS MINISTROS DE ESTADO DO PLANEJAMENTO': 25,
 'ORÇAMENTO E GESTÃO': 3150,
 'DA FAZENDA E DA DEFESA': 3,
 'MARTUS TAVARES M': 2,
 'PEDRO MALAN\nM': 1,
 'GERALDO MAGELA DA CRUZ QUINTÃO\nM': 5,
 'ANEXO\nEMPRESA BRASILEIRA DE AERONÁUTICA S': 2,
 'EMBRAER\nPROCESSO INTERESSADO PROCESSO ANTERIOR': 1,
 'EDISON BOTOSSI CARDOSO': 1,
 'RONEY JOSE FERREIRA': 1,
 'O DIRETOR DO CENTRO TECNOLÓGICO DA MARINHA EM 

In [11]:
contagem = pl.DataFrame(
    {
        'entidades': list(contagem_entidades.keys()),
        'contagem': list(contagem_entidades.values())
    }
).sort('contagem', descending=True)

In [ ]:
maybe_matches = {}
ents = contagem.get_column('entidades').to_list()

for i, ent in enumerate(ents[:-1]):
    print(i, end='\r')
    tail = ents[i + 1:]
    scores = process.cdist(
        [ent],
        tail,
        scorer=rapidfuzz_fuzz.ratio,
        score_cutoff=81,
        dtype=np.uint8,
        workers=-1,
    )[0]
    matches = [candidate for candidate, score in zip(tail, scores) if score > 80]
    if matches:
        maybe_matches[ent] = matches
